<a href="https://colab.research.google.com/github/MohdFuzailHaider/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Loading dataset

In [50]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
import pandas as pd
from huggingface_hub import hf_hub_download

In [51]:
fact_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-06/data_0.parquet",
    token=HF_TOKEN
)
print(fact_path)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-06/data_0.parquet


In [52]:
dim_content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet",
    token=HF_TOKEN
)

print(dim_content_path)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_content.parquet


In [53]:
fact_df = pd.read_parquet(fact_path)
dim_content_df = pd.read_parquet(dim_content_path)

In [54]:
print("Fact table shape:", fact_df.shape)
print("Dim content shape:", dim_content_df.shape)

Fact table shape: (11694072, 30)
Dim content shape: (519606, 26)


In [55]:
display(fact_df.head())
display(dim_content_df.head())

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-06-01,client_3ffa76342f366962,content_cde79a1a7432ce40,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2026-06-01,client_3ffa76342f366962,content_bbd33968edccaf24,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-06-01,client_3ffa76342f366962,content_d41105eaa19670ea,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-06-01,client_3ffa76342f366962,content_902b2d9b3d8a19a2,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2026-06-01,client_3ffa76342f366962,content_e70ae5bf6ab35b59,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682.0,2555.0,None,None,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438.0,2430.0,None,None,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576.0,2645.0,None,None,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457.0,2522.0,None,None,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776.0,2552.0,None,None,True,False


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule:**
* Pages will be prioritized for content refresh when they show a combination of high search visibility and content staleness.
* Pages that are stale but still receive meaningful search impressions may represent valuable opportunities for review because improving them could potentially affect pages that already have search visibility.
* The rule provides a prioritized review queue and does not guarantee that refreshing a page will improve its performance. \
**Reason:**
* stale_visible_page — The page is stale and has meaningful search visibility, so it is prioritized for refresh review.
* visible_page — The page has search visibility but does not show a strong staleness signal, so it may be monitored rather than immediately refreshed.
* low_priority — The page does not currently show enough combined evidence of staleness and visibility to receive high refresh priority.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [56]:
daily_df = fact_df[["report_date","client_hash_id","content_hash_id","gsc_impressions"]].copy()
daily_df["report_date"] = pd.to_datetime(daily_df["report_date"])

In [57]:
daily_df["period"] = daily_df["report_date"].apply(
    lambda x: "early" if x.day <= 15 else "late")

period_df = (daily_df.groupby(["client_hash_id", "content_hash_id", "period"],as_index=False)
    .agg(impressions=("gsc_impressions", "sum")))

period_df.head()

,client_hash_id,content_hash_id,period,impressions
0,client_04660893ae39614a,content_004de9653278b5a4,early,0
1,client_04660893ae39614a,content_004de9653278b5a4,late,0
2,client_04660893ae39614a,content_00dc5efae381b2ab,early,0
3,client_04660893ae39614a,content_00dc5efae381b2ab,late,0
4,client_04660893ae39614a,content_01410f2556c327ac,early,0


In [58]:
comparison_df = (period_df.pivot(
        index=["client_hash_id", "content_hash_id"],columns="period",values="impressions"
    ).fillna(0).reset_index())

comparison_df.head()

period,client_hash_id,content_hash_id,early,late
0,client_04660893ae39614a,content_004de9653278b5a4,0.0,0.0
1,client_04660893ae39614a,content_00dc5efae381b2ab,0.0,0.0
2,client_04660893ae39614a,content_01410f2556c327ac,0.0,0.0
3,client_04660893ae39614a,content_019f27f634053ca7,0.0,0.0
4,client_04660893ae39614a,content_01efa71faea45dcc,0.0,0.0


In [59]:
comparison_df["decline_pct"] = ((comparison_df["early"] - comparison_df["late"])
    / comparison_df["early"].replace(0, 1)) * 100

comparison_df.head()

period,client_hash_id,content_hash_id,early,late,decline_pct
0,client_04660893ae39614a,content_004de9653278b5a4,0.0,0.0,0.0
1,client_04660893ae39614a,content_00dc5efae381b2ab,0.0,0.0,0.0
2,client_04660893ae39614a,content_01410f2556c327ac,0.0,0.0,0.0
3,client_04660893ae39614a,content_019f27f634053ca7,0.0,0.0,0.0
4,client_04660893ae39614a,content_01efa71faea45dcc,0.0,0.0,0.0


In [60]:
comparison_df["is_declining"] = (
    (comparison_df["early"] > 0) &
    (comparison_df["late"] < comparison_df["early"])
)

comparison_df["is_declining"].value_counts()

,count
is_declining,
False,289629
True,119576


In [61]:
# Keep pages with a measurable decline and meaningful early impressions
baseline_df = comparison_df[
    (comparison_df["is_declining"]) &
    (comparison_df["early"] >= 10)
].copy()

# Higher score = more severe decline
baseline_df["baseline_score"] = (baseline_df["early"] - baseline_df["late"])
baseline_df[
    ["content_hash_id", "early", "late", "decline_pct", "baseline_score"]
].head()

period,content_hash_id,early,late,decline_pct,baseline_score
1219,content_0059a4d4195810c9,688.0,157.0,77.180233,531.0
1220,content_005b6b7f7b8dda7f,572.0,385.0,32.692308,187.0
1221,content_0094c7d0fbcc07b7,95.0,52.0,45.263158,43.0
1225,content_0153b7dedc3fc40d,562.0,133.0,76.334520,429.0
1227,content_01ad5f3e74c28a0d,481.0,316.0,34.303534,165.0


In [62]:
df = baseline_df.copy()

df["reason_code"] = "DECLINING_IMPRESSIONS"
df["action"] = "REFRESH"
df["rank"] = (df["baseline_score"]
    .rank(method="first", ascending=False).astype(int))

baseline_queue = df[
    ["content_hash_id", "baseline_score", "rank", "reason_code", "action"]
].sort_values("rank")

baseline_queue.head()

period,content_hash_id,baseline_score,rank,reason_code,action
376256,content_963de14b1f58978f,220288.0,1,DECLINING_IMPRESSIONS,REFRESH
292420,content_943dc881428182b8,127376.0,2,DECLINING_IMPRESSIONS,REFRESH
293275,content_b902320872acab45,114796.0,3,DECLINING_IMPRESSIONS,REFRESH
378227,content_cc26620b2cbb837f,105479.0,4,DECLINING_IMPRESSIONS,REFRESH
34840,content_11bf4c33adea7bdc,90279.0,5,DECLINING_IMPRESSIONS,REFRESH


In [63]:
import os

os.makedirs("work/outputs", exist_ok=True)

baseline_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

CSV saved successfully!


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [64]:
top20 = (df.sort_values("rank").head(20)[
        ["content_hash_id","early","late","decline_pct","baseline_score","rank","reason_code","action"]])

top20

period,content_hash_id,early,late,decline_pct,baseline_score,rank,reason_code,action
376256,content_963de14b1f58978f,417650.0,197362.0,52.744643,220288.0,1,DECLINING_IMPRESSIONS,REFRESH
292420,content_943dc881428182b8,209896.0,82520.0,60.685292,127376.0,2,DECLINING_IMPRESSIONS,REFRESH
293275,content_b902320872acab45,174849.0,60053.0,65.654365,114796.0,3,DECLINING_IMPRESSIONS,REFRESH
378227,content_cc26620b2cbb837f,165448.0,59969.0,63.753566,105479.0,4,DECLINING_IMPRESSIONS,REFRESH
34840,content_11bf4c33adea7bdc,98747.0,8468.0,91.424550,90279.0,5,DECLINING_IMPRESSIONS,REFRESH
78501,content_68ffc95c19f61219,93517.0,4223.0,95.484244,89294.0,6,DECLINING_IMPRESSIONS,REFRESH
300181,content_adcc7b85a04c187d,184064.0,99441.0,45.974770,84623.0,7,DECLINING_IMPRESSIONS,REFRESH
293825,content_d0acf7062bc6b257,109748.0,35274.0,67.859095,74474.0,8,DECLINING_IMPRESSIONS,REFRESH
299116,content_54f2b96801c90591,167627.0,99441.0,40.677218,68186.0,9,DECLINING_IMPRESSIONS,REFRESH
285140,content_26be6eb87f2bc45f,61575.0,2115.0,96.565164,59460.0,10,DECLINING_IMPRESSIONS,REFRESH


**Rank 1** — content_963de14b1f58978f
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — impressions declined from 417,650 to 197,362, representing 220,288 observed impressions lost.
* What would make it wrong: The decline could be caused by seasonality, a temporary search change, or external factors that a content refresh would not fix.\
**Rank 2** — content_943dc881428182b8
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — impressions declined from 209,896 to 82,520, representing 127,376 observed impressions lost.
* What would make it wrong: The observed decline may be caused by changes outside the content itself, such as search demand or ranking changes.\
Rank 3 — content_b902320872acab45
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — impressions declined from 174,849 to 60,053, representing 114,796 observed impressions lost.
* What would make it wrong: The page may have experienced a temporary or seasonal decline rather than a problem that requires refreshing.\
Rank 4 — content_cc26620b2cbb837f
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — impressions declined from 165,448 to 59,969, representing 105,479 observed impressions lost.
* What would make it wrong: External changes in search demand or competition could explain the decline.\
Rank 5 — content_11bf4c33adea7bdc
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — impressions declined from 98,747 to 8,468, a 91.4% observed decline.
* What would make it wrong: The decline may be caused by factors unrelated to content freshness.\
Rank 6 — content_68ffc95c19f61219
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — impressions declined from 93,517 to 4,223, a 95.5% observed decline.
* What would make it wrong: A technical issue or external ranking change could be responsible instead of stale content.\
Rank 7 — content_adcc7b85a04c187d
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — the page lost 84,623 impressions between the two periods.
* What would make it wrong: The decline could reflect changes in search demand rather than content quality.\
Rank 8 — content_d0acf7062bc6b257
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — impressions declined from 109,748 to 35,274.
* What would make it wrong: The page could be affected by external ranking or market changes.\
Rank 9 — content_54f2b96801c90591
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: Medium-High — the page lost 68,186 impressions, although the percentage decline was lower than several other top-ranked pages.
* What would make it wrong: The absolute-loss scoring may prioritize large pages even when their relative decline is less severe.\
Rank 10 — content_26be6eb87f2bc45f
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — impressions declined from 61,575 to 2,115, representing a 96.6% decline.
* What would make it wrong: The decline could be temporary or caused by factors that refreshing cannot solve.\
Rank 11 — content_9ef3d7516483e665
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: Medium-High — the page lost 58,665 impressions.
* What would make it wrong: The relative decline is only 35.7%, so the page may still be performing strongly despite the large absolute loss.\
Rank 12 — content_62aa9910370a05d3
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — impressions declined from 64,478 to 7,705, an 88.1% decline.
* What would make it wrong: Search demand or ranking changes could explain the observed decline.\
Rank 13 — content_c1f764a2f362d1c3
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: Medium-High — the page lost 49,347 impressions.
* What would make it wrong: Its 34.7% relative decline may not necessarily indicate that the content needs refreshing.\
Rank 14 — content_33d31496fca9665e
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: Medium-High — the page lost 47,841 impressions.
* What would make it wrong: The decline could be caused by changing demand rather than content freshness.\
Rank 15 — content_7471467133493ce6
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — impressions declined by 47,172, representing a 76.8% decline.
* What would make it wrong: External search or technical factors may be responsible.\
Rank 16 — content_32c5cc913fb4ff41
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: Medium — although the page lost 46,602 impressions, the relative decline was 36.0%.
* What would make it wrong: The page may still perform well enough that refreshing is not the highest-priority action.\
Rank 17 — content_661a7734f691bef5
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — the page lost 45,332 impressions, with a 59.6% observed decline.
* What would make it wrong: The decline may be temporary or unrelated to content quality.\
Rank 18 — content_4c15fe8dc370f3cd
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — the page lost 42,044 impressions, representing a 71.1% decline.
* What would make it wrong: Search demand changes could make a refresh ineffective.\
Rank 19 — content_ba462518dad435fc
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — impressions declined from 65,689 to 24,850, losing 40,839 impressions.
* What would make it wrong: The observed decline could be driven by external ranking or demand changes.\
Rank 20 — content_14824df843e76fa8
* Action: REFRESH
* Reason code: DECLINING_IMPRESSIONS
* Confidence: High — the page lost 39,246 impressions, representing a 74.4% decline.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [65]:
print(baseline_df.columns.tolist())
check_score = baseline_df["early"] - baseline_df["late"]

(check_score == baseline_df["baseline_score"]).all()

['client_hash_id', 'content_hash_id', 'early', 'late', 'decline_pct', 'is_declining', 'baseline_score']


np.True_

In [66]:
suspicious_columns = [
    col for col in baseline_df.columns
    if any(word in col.lower() for word in [
        "flag", "label", "target", "future", "outcome", "action"
    ])
]

print(suspicious_columns)

[]


In [67]:
daily_df.groupby("period")["report_date"].agg(
    ["min", "max", "count"]
)

,min,max,count
period,,,
early,2026-06-01,2026-06-15,5934916
late,2026-06-16,2026-06-30,5759156


**Weak picks**
* The weaker picks are pages where the rule may detect a decline but cannot explain why the decline happened.

* My rule only measures the difference in impressions between the early and late periods. Therefore, a page can be ranked highly even if the decline was caused by seasonality, reduced search demand, ranking changes, or other factors unrelated to outdated content.

* For this reason, REFRESH is a recommendation for human review, not proof that refreshing the page is the correct solution.

**Leakage check**

* I verified that the baseline dataframe does not contain product flags, target labels, outcome variables, or action labels as scoring inputs.

* The score uses only:
    baseline_score = early impressions − late impressions

The comparison uses:
* Early: 2026-06-01 to 2026-06-15
* Late: 2026-06-16 to 2026-06-30

All inputs come from the defined observation window, and no separate future outcome window was used. Therefore, I found no obvious label-derived or product-flag leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ YES ] Every section above is filled — markdown thinking AND the code that backs it
- [ YES ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ YES ] No client names, URLs, or private queries anywhere
- [ YES ] My claims use careful words: observed, measured, directional, decision-support
- [ YES ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.